## Imports

In [47]:
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import torchvision
import torchvision.transforms as transforms

device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(device)

cuda


## Transform
`ToTensor` scales pixels to [0, 1], then `Normalize` maps them to [-1, 1].

In [48]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

## Load datasets

In [49]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    root = '/content/drive/MyDrive/cifar_data'
except ImportError:
    root = '../cifar_data'  # running locally

train_data = torchvision.datasets.CIFAR10(root=root, train=True, transform=transform, download=False)
test_data = torchvision.datasets.CIFAR10(root=root, train=False, transform=transform, download=False)


## Data loaders

In [50]:
train_loader = torch.utils.data.DataLoader(train_data, batch_size=32, num_workers=2, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=32, num_workers=2, shuffle=False)

## Inspect one sample

In [51]:
image, label = train_data[0]
image.size()

torch.Size([3, 32, 32])

## Class names

In [52]:
class_names = train_data.classes
class_names

['airplane',
 'automobile',
 'bird',
 'cat',
 'deer',
 'dog',
 'frog',
 'horse',
 'ship',
 'truck']

In [53]:
class NeuralNet(nn.Module):
    
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 12, 5, padding=2)    # 3x32x32  -> 12x32x32
        self.skip1 = nn.Conv2d(3, 12, 1)
        self.pool = nn.MaxPool2d(2, 2)                 # 12x32x32 -> 12x16x16
        
        self.conv2 = nn.Conv2d(12, 24, 5, padding=2)   # 12x16x16 -> 24x16x16
        self.skip2 = nn.Conv2d(12, 24, 1)
        self.pool2 = nn.AvgPool2d(2, 2)                # 24x16x16 -> 24x8x8
        
        self.fc1 = nn.Linear(24 * 8 * 8, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)
    
    def forward(self, x):
        x = F.leaky_relu(self.conv1(x) + self.skip1(x))
        x = self.pool(x)
        x = F.leaky_relu(self.conv2(x) + self.skip2(x))
        x = self.pool2(x)
        x = torch.flatten(x, 1)
        x = F.leaky_relu(self.fc1(x))
        x = F.leaky_relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [54]:
net = NeuralNet().to(device)
loss_function = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

EPOCHS = 30
for epoch in range(EPOCHS):
    print(f"training epoch {epoch}")
    
    running_loss = 0.0
    
    for i, data in enumerate(train_loader):
        inputs, labels = data
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        
        outputs = net(inputs)
        
        loss = loss_function(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    print(f"loss: {running_loss/len(train_loader)}")

training epoch 0
loss: 2.0477363036674943
training epoch 1
loss: 1.5615108279333767
training epoch 2
loss: 1.3834801368124579
training epoch 3
loss: 1.2577707269065135
training epoch 4
loss: 1.1653534446247709
training epoch 5
loss: 1.0906628288676627
training epoch 6
loss: 1.0284493554500305
training epoch 7
loss: 0.9736541983834155
training epoch 8
loss: 0.9219721418233995
training epoch 9
loss: 0.8748844023209997
training epoch 10
loss: 0.8353546209352106
training epoch 11
loss: 0.7900289295197144
training epoch 12
loss: 0.7516674566215532
training epoch 13
loss: 0.7125953106413419
training epoch 14
loss: 0.674970616687206
training epoch 15
loss: 0.6383610782776593
training epoch 16
loss: 0.6019932619002257
training epoch 17
loss: 0.5663859322905465
training epoch 18
loss: 0.5311555818190425
training epoch 19
loss: 0.4994144177091709
training epoch 20
loss: 0.46210347014459674
training epoch 21
loss: 0.4338638406063377
training epoch 22
loss: 0.4036060427044419
training epoch 23
los

In [55]:
torch.save(net.state_dict(),'trained_net.pth')

In [56]:
net = NeuralNet()
net.load_state_dict(torch.load('trained_net.pth'))

<All keys matched successfully>

In [61]:
correct = 0
total = 0

net.eval()

with torch.no_grad():
    for data in test_loader:
        image,labels = data 
        outputs = net(image)
        _,predicted = torch.max(outputs,1)
        total += labels.size(0)
        correct += (predicted==labels).sum().item()
accuracy = 100*correct/total
print(f"accuracy is {accuracy}")
        

accuracy is 68.87


In [ ]:
new_transform = transforms.Compose([
    transforms.Resize((32,32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5)),
])


AttributeError: 'Resize' object has no attribute 'transforms'